# Fitting a curve with `Numeric`

`Numeric.CurveFit` is MINPACK's Levenberg-Marquardt, held to scipy's
`curve_fit` to the bit. The model is a procedure the notebook writes:
a function of one real and a parameter vector, `y = f (t; p)`.

The state cell makes a noisy sine; the program after it fits
`p[0] * sin (p[1] * t + p[2]) + p[3]` to it and draws both.

In [ ]:
STATEFUL DEFINITION MODULE Signal ;

VAR t, y : SLICE OF F64 ;

END Signal.

IMPLEMENTATION MODULE Signal ;

IMPORT Io ;
IMPORT Math ;
IMPORT Stats ;

VAR
  st : Stats.Stream ;
  i : I64 ;

BEGIN
  st := Stats.Seed (7) ;
  t := NEW (F64, 200) ;
  y := NEW (F64, 200) ;
  FOR i := 0 TO 199 DO
    t[i] := F64 (i) * 0.05 ;
    y[i] := 2.0 * Math.Sin (1.5 * t[i] + 0.3) + 0.5 + 0.2 * Stats.Normal (st)
  END
EXCEPT
| ValueRange :
    Io.ErrLine ('a value out of range')
END Signal.

In [ ]:
MODULE Fit ;

IMPORT Fmt ;
IMPORT Faults ;
IMPORT Io ;
IMPORT Math ;
IMPORT NbCells ;
IMPORT Numeric ;
IMPORT Plot ;
IMPORT Signal ;

(* the model: a function of t with its parameters beside it, the
   shape Numeric.Fn asks for *)
PROCEDURE Model (t: F64 ; RO p: SLICE OF F64) : F64 RAISES ValueRange =
BEGIN
  RETURN p[0] * Math.Sin (p[1] * t + p[2]) + p[3]
END Model ;

VAR
  pool : POOL ;
  p, cov, fit : SLICE OF F64 ;
  r : Numeric.Fitted ;
  i : I64 ;

BEGIN
  p := NEW (pool, F64, 4) ;
  p[0] := 1.0 ; p[1] := 1.3 ; p[2] := 0.0 ; p[3] := 0.0 ;   (* the start *)
  cov := NEW (pool, F64, 16) ;
  r := Numeric.CurveFit (Model, Signal.t, Signal.y, p, cov) ;
  Io.WriteLine ('stopped for MINPACK reason ' + Fmt.I64Str (r.info) + ' after '
                + Fmt.I64Str (r.nfev) + ' evaluations, sum of squares '
                + Fmt.Fixed (r.cost, 4)) ;
  FOR i := 0 TO 3 DO
    Io.WriteLine ('  p[' + Fmt.I64Str (i) + '] = ' + Fmt.Fixed (p[i], 4)
                  + ' +- ' + Fmt.Fixed (Math.Sqrt (cov[i * 4 + i]), 4))
  END ;
  fit := NEW (pool, F64, LEN (Signal.t)) ;
  FOR i := 0 TO LEN (Signal.t) - 1 DO fit[i] := Model (Signal.t[i], p) END ;
  Plot.ClearFigure () ;
  Plot.SetDots (Signal.t, Signal.y) ;
  Plot.AddLine (Signal.t, fit, 1, 'fit') ;
  Io.WriteFile ('fit.svg', Plot.Render ('a sine fitted to noisy data', 't', 'y')) ;
  NbCells.PutF64s ('params', p)
EXCEPT
| Numeric.NoConverge :
    Io.ErrLine ('the fit did not converge')
| Faults.SizeError (got, want) :
    Io.ErrLine ('the sizes disagree')
| Faults.BadArg (what) :
    Io.ErrLine ('bad argument: ' + what)
| ValueRange :
    Io.ErrLine ('a value out of range')
| Overflow :
    Io.ErrLine ('overflow')
| IndexError :
    Io.ErrLine ('out of range')
| Io.IOError (path) :
    Io.ErrLine ('cannot write ' + path)
END Fit.

The fitted parameters were stored by name. With the same model,
`Numeric.Root` (Brent) finds where the fitted curve crosses its own
offset, and `Numeric.Integral` (adaptive Gauss-Kronrod) integrates it
over a period -- both are scipy's algorithms, transcribed.

In [ ]:
MODULE Analyse ;

IMPORT Fmt ;
IMPORT Faults ;
IMPORT Io ;
IMPORT Math ;
IMPORT NbCells ;
IMPORT Numeric ;

PROCEDURE Model (t: F64 ; RO p: SLICE OF F64) : F64 RAISES ValueRange =
BEGIN
  RETURN p[0] * Math.Sin (p[1] * t + p[2]) + p[3]
END Model ;

(* the curve less its offset: zero where the sine is *)
PROCEDURE Centred (t: F64 ; RO p: SLICE OF F64) : F64 RAISES ValueRange =
BEGIN
  RETURN Model (t, p) - p[3]
END Centred ;

VAR
  pool : POOL ;
  p : SLICE OF F64 ;
  t0, period, area, err : F64 ;

BEGIN
  p := NbCells.GetF64s (pool, 'params') ;
  period := 2.0 * Math.Pi / p[1] ;
  t0 := Numeric.Root (Centred, p, 1.0, 1.0 + period / 2.0, 1.0e-10) ;
  Io.WriteLine ('period      ' + Fmt.Fixed (period, 4)) ;
  Io.WriteLine ('first zero  ' + Fmt.Fixed (t0, 6) + ' after t = 1') ;
  area := Numeric.Integral (Model, p, t0, t0 + period, 1.0e-10, 1.0e-10, err) ;
  Io.WriteLine ('integral over one period ' + Fmt.Fixed (area, 6)
                + '  (offset times period: ' + Fmt.Fixed (p[3] * period, 6) + ')')
EXCEPT
| NbCells.Missing (name) :
    Io.ErrLine ('nothing stored as ' + name + ': run the fit first')
| NbCells.WrongType (name, stored, wanted) :
    Io.ErrLine (name + ' holds ' + stored)
| Numeric.NoBracket :
    Io.ErrLine ('no sign change in the bracket')
| Numeric.NoConverge :
    Io.ErrLine ('no convergence')
| Faults.BadArg (what) :
    Io.ErrLine ('bad argument: ' + what)
| Io.IOError (path) :
    Io.ErrLine ('cannot read ' + path)
| ValueRange :
    Io.ErrLine ('a value out of range')
| Overflow :
    Io.ErrLine ('overflow')
| IndexError :
    Io.ErrLine ('out of range')
END Analyse.